In [9]:
import os

from dotenv import load_dotenv
from collections import Counter
from langchain_cohere import ChatCohere
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, Markdown
from langchain_core.runnables import RunnableParallel, RunnableLambda

load_dotenv()

True

In [2]:
model = ChatCohere(
    cohere_api_key=os.getenv("COHERE_API_KEY"),
    model="command-a-03-2025",
    temperature=0.7,
)

In [3]:


# طراحی پرامپت CoT
prompt = ChatPromptTemplate.from_template("""
مسئله زیر را حل کن. حتماً مرحله به مرحله فکر کن و در خط آخر فقط عدد نهایی را بنویس.
مسئله: {question}
""")

# ساخت یک زنجیره واحد
# این زنجیره "یک بار" فکر می‌کند و جواب می‌دهد
single_chain = prompt | model | StrOutputParser()

In [4]:
# ورودی {question} به هر سه شاخه می‌رود
consistency_chain = RunnableParallel({
    "path_1": single_chain,
    "path_2": single_chain,
    "path_3": single_chain
})

In [5]:
def majority_vote(results: dict):
    answers = list(results.values())
    # نکته: در پروژه واقعی باید "جواب نهایی" را از متن استخراج و نرمال‌سازی کنید
    # اینجا برای سادگی فرض می‌کنیم تشابه متن کافیست
    vote_counts = Counter(answers)
    most_common, count = vote_counts.most_common(1)[0]
    
    return f"\n>>> پاسخ نهایی (با {count} رای موافق):\n{most_common}"


In [6]:

# اضافه کردن رای‌گیری به زنجیره نهایی
final_chain = consistency_chain | RunnableLambda(majority_vote)

In [10]:
question = "I am 6 years old. My sister is half my age. When I am 70, how old will my sister be?"

output = final_chain.invoke({"question": question})


In [11]:
answer = (
    output
    .replace(r"\[", "\n$$\n")
    .replace(r"\]", "\n$$\n")
    .replace(r"\(", "$")
    .replace(r"\)", "$")
)

rtl_answer = f"""
<div dir="rtl" style="text-align: right; line-height: 2;">

{answer}

</div>
"""

display(Markdown(rtl_answer))


<div dir="rtl" style="text-align: right; line-height: 2;">


>>> پاسخ نهایی (با 1 رای موافق):
مرحله به مرحله حل می‌کنیم:

1. **سن فعلی من**: 6 سال  
2. **سن فعلی خواهرم**: نصف سن من = 6 / 2 = 3 سال  
3. **تفاوت سنی**: 6 - 3 = 3 سال (همیشه ثابت است)  
4. **وقتی من 70 ساله باشم**: سن خواهرم = 70 - 3 = 67 سال  

**67**

</div>
